spliting Data

In [ ]:
import os
import random
import shutil

image_source = 'images'
label_source = 'labels-YOLO'
base_dir = 'datasets'

for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

images = [f for f in os.listdir(image_source) if f.endswith(('.jpg', '.jpeg', '.png'))]
random.shuffle(images)

split_idx = int(len(images) * 0.8)
train_images = images[:split_idx]
val_images = images[split_idx:]

def move_files(files, subset):
    for f in files:

        shutil.copy(os.path.join(image_source, f), os.path.join(base_dir, f'images/{subset}', f))

        label_file = f.rsplit('.', 1)[0] + '.txt'
        if os.path.exists(os.path.join(label_source, label_file)):
            shutil.copy(os.path.join(label_source, label_file), os.path.join(base_dir, f'labels/{subset}', label_file))


move_files(train_images, 'train')
move_files(val_images, 'val')

print(f"Done! Train: {len(train_images)}, Val: {len(val_images)}")

Done! Train: 1607, Val: 402


Yaml

In [ ]:
%%writefile data.yaml

path: ./datasets
train: images/train
val: images/val

nc: 3
names: ['Pothole', 'Crack', 'Other']

Overwriting data.yaml


Traning


In [ ]:
from ultralytics import YOLO


model = YOLO('yolov8s.pt')


results = model.train(
    data='data.yaml',
    epochs=100,
    imgsz=640,
    batch=8,
    device='mps',
    workers=2,
    patience=20,
    project='RoadDamage',
    name='m2_training',
    exist_ok=True,
    augment=True
)

print(" Training Finished!")

Ultralytics 8.4.46 🚀 Python-3.13.0 torch-2.11.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=m2_training, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspective=0.0, plots=True, pose=12.

In [12]:


print("  ROAD DAMAGE DETECTION ")

print("\nOVERALL METRICS:")
print(f"   {'mAP50':<15} : 0.9290  (92.9%)")
print(f"   {'mAP50-95':<15} : 0.6190  (61.9%)")
print(f"   {'Precision':<15} : 0.9040  (90.4%)")
print(f"   {'Recall':<15} : 0.8630  (86.3%)")


classes = [
    ("Pothole", 0.866, 0.782, 0.888, 0.561),
    ("Crack",   0.846, 0.873, 0.926, 0.658),
    ("Other",   0.922, 0.895, 0.962, 0.648),
    ("All",     0.878, 0.850, 0.926, 0.622),
]


print("\n MODEL INFO:")
print(f"   {'Parameters':<15} : 11,126,745")
print(f"   {'Layers':<15} : 73")
print(f"   {'GFLOPs':<15} : 28.4")
print(f"   {'Classes':<15} : Pothole, Crack, Other")
print(f"   {'Image size':<15} : 640 x 640")
print(f"   {'Device':<15} : Apple M2 (MPS)")



  ROAD DAMAGE DETECTION 

OVERALL METRICS:
   mAP50           : 0.9290  (92.9%)
   mAP50-95        : 0.6190  (61.9%)
   Precision       : 0.9040  (90.4%)
   Recall          : 0.8630  (86.3%)

 MODEL INFO:
   Parameters      : 11,126,745
   Layers          : 73
   GFLOPs          : 28.4
   Classes         : Pothole, Crack, Other
   Image size      : 640 x 640
   Device          : Apple M2 (MPS)
